# Midterm 1: run the data generator in Colab

Use this notebook if you cannot run Python or git on your own machine. It only **generates and shows your data**. You write your answers in `ANSWERS.md` on **your fork on github.com** (open the file, click the pencil icon, edit, *Commit changes*), then submit the commit URL in Canvas. See the README for the rules.

1. Fill in `GITHUB_USERNAME` (the account that owns your fork) and `CODE` (`PRACTICE` to rehearse; the real code is on the board during the exam).
2. Run the three code cells in order.

In [2]:
GITHUB_USERNAME = "tomiris-nurbolat"   # e.g. "aigerim-k"
CODE = "PRACTICE"      # replace with the session code during the exam
assert GITHUB_USERNAME, "fill in GITHUB_USERNAME first"

In [3]:
import os

if not os.path.exists("midterm-1-AI-course"):
    !git clone --quiet https://github.com/unreal-kz/midterm-1-AI-course.git
%cd midterm-1-AI-course
!python make_my_data.py --github {GITHUB_USERNAME} --code {CODE}

/content/midterm-1-AI-course
Wrote 3 files to my_data/ · fingerprint bfd9a65e
Next: open my_data/DATA.md


In [4]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(open("my_data/DATA.md", encoding="utf-8").read()))
ranks = pd.read_csv("my_data/retrieval_ranks.csv")   # Part 1
docs = pd.read_csv("my_data/doc_results.csv")        # Part 2
display(ranks.head(), docs)

# Midterm 1: your data

GitHub username: `tomiris-nurbolat` · session code: `practice` · generator v1

**Fingerprint: `bfd9a65e`**: copy it into the first lines of `ANSWERS.md`. All numbers below are synthetic: this is an exam, not a measurement.

## Part 1: inputs

`retrieval_ranks.csv`: 60 test questions (20 EN, 20 RU, 20 KK). `rank_A` and `rank_B` give the position of the correct chunk in the top 10 retrieved under config A and config B; **11 means the correct chunk was not in the top 10.**

Production traffic and Model M token counts (tokens per item, as counted by Model M's tokenizer):

| language | share of traffic | question tokens | answer tokens | chunk A tokens | chunk B tokens |
|---|---|---|---|---|---|
| EN | 26 % | 32 | 144 | 264 | 79 |
| RU | 35 % | 73 | 327 | 599 | 179 |
| KK | 39 % | 100 | 448 | 821 | 246 |

- Requests per day: **6,500**, in the traffic shares above.
- System prompt: **194** tokens in every request.
- Price: **$2** per million input tokens, **$12** per million output tokens.
- Annual budget for this service (model calls only): **$21,900**.

## Part 2: inputs

`doc_results.csv`: for each pipeline, language and scan condition, `n_docs` test invoices and `n_correct` = invoices whose number, date, seller ID and total were all extracted exactly (strict scorer, as in Lab 04).

- **Pipeline H**: hosted vision-language API in a data centre abroad, pay per token; each page is sent as an image.
- **Pipeline L**: open vision-language model on your own server in Kazakhstan, fixed monthly cost.

- Volume: **5,500** scanned pages per day. In **9 %** of them an individual's name and ИИН are printed (sole proprietors, receipts).
- Page image: **827 × 1169** pixels. Pipeline H counts an image as ⌈width ÷ 28⌉ × ⌈height ÷ 28⌉ input tokens and returns about **359** output tokens per page (the JSON).
- Pipeline H price: **$2** per million input tokens, **$10** per million output tokens.
- Pipeline L: one server costs **$900** per month and handles up to **6,000** pages per day; add servers as needed.

## Part 3: your five questions

Answer each in `ANSWERS.md`: one **Number** line and one **Why** line of at most 40 words.

### Q1

The same support request, once in English and once in Kazakh, through Model M ($1 per million input tokens, $5 per million output tokens):

| | input tokens | output tokens |
|---|---|---|
| EN | 1,796 | 105 |
| KK | 4,843 | 210 |

Compute the token ratio (total tokens KK ÷ EN) and the bill ratio (cost KK ÷ cost EN). **Number:** the bill ratio (2 decimals). **Why** (at most 40 words): why is the bill ratio not equal to the token ratio?

### Q2

A teammate sets the chunk size to **512 tokens, counted with cl100k**. The retrieval model (e5-small) reads at most 512 of *its own* tokens. Assume e5 tokens grow in proportion to cl100k tokens at the rate measured in Lecture 4.1 (a chunker told 256 cl100k tokens produced chunks of 263 / 128 / 76 e5 tokens for EN / RU / KK), and that every chunk is exactly the full size.

**Number:** the share of an English chunk, in percent (1 decimal), that the embedding never sees. **Why** (at most 40 words): which of the three languages are cut, and what happens to the cut part?

### Q3

A language model gives these next-token scores for the prompt "The branch is closed on":

| token | score |
|---|---|
| Monday | 4.2 |
| Friday | 2.8 |
| Sunday | 2.3 |
| holiday | 1.3 |
| Easter | 0.9 |

Apply softmax at temperature T = 0.5. **Number:** the probability of the top token, in percent (1 decimal). **Why** (at most 40 words): what did changing T do to the ranking of the tokens, and to the set of tokens that can be sampled?

### Q4

Two models answer the same 100 factual questions (1 point per correct answer on the public leaderboard):

| | correct | wrong | "I don't know" |
|---|---|---|---|
| Model X | 70 | 30 | 0 |
| Model Y | 64 | 24 | 12 |

Now score them again with a penalty: +1 per correct answer, −1 per wrong answer, 0 for "I don't know". **Number:** Y's penalised score minus X's (negative if X is ahead). **Why** (at most 40 words): which model does the public leaderboard prefer, and what does that teach a model to do?

### Q5

A support chat adds **95 new tokens per turn** (customer message plus assistant reply) and has **23 turns**. The model keeps no state, so the full history is sent again on every turn (Lecture 1.2). As in the Lecture 1.2 table (100, 200, 300, ... tokens billed on turns 1, 2, 3, ...), turn *i* is billed for the tokens of all *i* turns so far, including its own. **Number:** the total input tokens billed over the whole conversation. **Why** (at most 40 words): how many times has the first message been paid for by the last turn, and what does prompt caching change?


,qid,lang,rank_A,rank_B
0,q01,EN,1,1
1,q02,KK,1,1
2,q03,EN,1,1
3,q04,RU,11,11
4,q05,EN,1,1


,pipeline,lang,condition,n_docs,n_correct
0,H,EN,clean,30,29
1,H,EN,degraded,30,27
2,H,RU,clean,30,28
3,H,RU,degraded,30,26
4,H,KK,clean,30,27
5,H,KK,degraded,30,25
6,L,EN,clean,30,27
7,L,EN,degraded,30,23
8,L,RU,clean,30,26
9,L,RU,degraded,30,21


In [8]:
# Midterm 1
import math
import make_my_data

data = make_my_data.generate(GITHUB_USERNAME, CODE)
p1 = data.params["part1"]
p2 = data.params["part2"]
p3 = data.params["part3"]

print("GitHub username:", GITHUB_USERNAME)
print("Session code:", CODE)
print("Fingerprint:", data.fingerprint)


# интервал Уилсона
def wilson(hits, n):
    z = 1.96
    p = hits / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


# PART 1
print()
print("PART 1:")

rows = p1["rows"]
n = len(rows)       # 60 вопросов

# 1a. recall и MRR
print()
print("1a")
for config in ["A", "B"]:
    if config == "A":
        col = 2
    else:
        col = 3

    hits1 = 0
    hits3 = 0
    hits5 = 0
    mrr_sum = 0
    for row in rows:
        rank = row[col]
        if rank <= 1:
            hits1 = hits1 + 1
        if rank <= 3:
            hits3 = hits3 + 1
        if rank <= 5:
            hits5 = hits5 + 1
        if rank <= 10:
            mrr_sum = mrr_sum + 1 / rank

    print("|", config, "|", round(hits1 / n, 3), "|", round(hits3 / n, 3), "|",
          round(hits5 / n, 3), "|", round(mrr_sum / n, 3), "|")

# 1b и 1c для каждого кандидата
candidates = [["A", 3], ["A", 5], ["B", 3], ["B", 5]]
langs = ["EN", "RU", "KK"]

print()
print("1b")
for cand in candidates:
    config = cand[0]
    k = cand[1]
    if config == "A":
        col = 2
    else:
        col = 3
    hits = 0
    for row in rows:
        if row[col] <= k:
            hits = hits + 1
    low, high = wilson(hits, n)
    print("|", config + ", k=" + str(k), "|", round(hits / n, 3), "|", round(low, 3), "|", round(high, 3), "|")

print()
print("1c")
for cand in candidates:
    config = cand[0]
    k = cand[1]
    cost_per_request = 0
    for lang in langs:
        tokens_in = p1["system_tokens"] + p1["question_tokens"][lang] + k * p1["chunk_tokens"][config][lang]
        tokens_out = p1["answer_tokens"][lang]
        cost = (tokens_in * p1["price_in"] + tokens_out * p1["price_out"]) / 1000000
        share = p1["share"][lang] / 100
        cost_per_request = cost_per_request + share * cost
    annual = cost_per_request * p1["requests_per_day"] * 365
    print("|", config + ", k=" + str(k), "|", round(annual), "|")

print()
print("Budget:", p1["budget"])

# recall по языкам
print()
print("Recall by language (for 1d)")
for cand in candidates:
    config = cand[0]
    k = cand[1]
    if config == "A":
        col = 2
    else:
        col = 3
    text = config + ", k=" + str(k) + ":"
    for lang in langs:
        hits = 0
        total = 0
        for row in rows:
            if row[1] == lang:
                total = total + 1
                if row[col] <= k:
                    hits = hits + 1
        low, high = wilson(hits, total)
        text = text + "  " + lang + " " + str(hits) + "/" + str(total)
        text = text + " [" + str(round(low, 2)) + ", " + str(round(high, 2)) + "]"
    print(text)


# PART 2
print()
print("PART 2:")

results = p2["results"]

# стоимость H
width = p2["page_px"][0]
height = p2["page_px"][1]
image_tokens = math.ceil(width / 28) * math.ceil(height / 28)
cost_h = p2["pages_per_day"] * 365 * (image_tokens * p2["price_in"] + p2["out_tokens_per_page"] * p2["price_out"]) / 1000000

# стоимость L
servers = math.ceil(p2["pages_per_day"] / p2["server_capacity"])
cost_l = servers * p2["server_month"] * 12

for pipeline in ["H", "L"]:
    correct = 0
    total = 0
    worst = None
    for row in results:
        if row[0] == pipeline:
            correct = correct + row[4]
            total = total + row[3]
            if worst is None or row[4] / row[3] < worst[4] / worst[3]:
                worst = row
    low, high = wilson(worst[4], worst[3])
    if pipeline == "H":
        cost = cost_h
    else:
        cost = cost_l
    print("|", pipeline, "|", round(correct / total, 3), "|", worst[1] + ", " + worst[2], "|",
          round(worst[4] / worst[3], 3), "|", round(low, 3), "|", round(high, 3), "|", round(cost), "|")

pii_pages = p2["pages_per_day"] * 365 * p2["pii_share"] / 100
print()
print("Personal-data pages per year:", round(pii_pages))

# все ячейки
print()
print("All cells (for 2b)")
for row in results:
    low, high = wilson(row[4], row[3])
    print(row[0], row[1], row[2], str(row[4]) + "/" + str(row[3]), "=", round(row[4] / row[3], 3),
          "[" + str(round(low, 3)) + ", " + str(round(high, 3)) + "]")


# PART 3
print()
print("PART 3:")

# Q1
q = p3["q1"]
if q["template"] == "a":
    char_ratio = q["chars"]["KK"] / q["chars"]["EN"]
    byte_ratio = q["bytes"]["KK"] / q["bytes"]["EN"]
    token_ratio = q["tokens"]["KK"] / q["tokens"]["EN"]
    print("Q1: chars", round(char_ratio, 2), "bytes", round(byte_ratio, 2), "tokens", round(token_ratio, 2))
    print("Q1 Number:", round(token_ratio / byte_ratio, 2))
else:
    token_ratio = (q["in_kk"] + q["out_kk"]) / (q["in_en"] + q["out_en"])
    cost_en = q["in_en"] * q["p_in"] + q["out_en"] * q["p_out"]
    cost_kk = q["in_kk"] * q["p_in"] + q["out_kk"] * q["p_out"]
    print("Q1: token ratio", round(token_ratio, 2))
    print("Q1 Number:", round(cost_kk / cost_en, 2))

# Q2
q = p3["q2"]
e5_en = q["size"] * 263 / 256
if q["template"] == "a":
    if e5_en > 512:
        lost = (e5_en - 512) / e5_en * 100
    else:
        lost = 0
    print("Q2: EN chunk =", round(e5_en, 1), "e5 tokens, RU =", q["size"] * 128 / 256, ", KK =", q["size"] * 76 / 256)
    print("Q2 Number:", round(lost, 1), "%")
else:
    start = q["frac"] / 100 * e5_en
    print("Q2: EN chunk =", round(e5_en, 1), "e5 tokens, sentence starts at", start)
    print("Q2 Number:", math.floor(start + 0.5))

# Q3
q = p3["q3"]
if q["template"] == "a":
    t = q["temperature"]
else:
    t = 1
exps = []
for score in q["logits"]:
    exps.append(math.exp(score / t))
total = sum(exps)
probs = []
for e in exps:
    probs.append(e / total)
print("Q3: probabilities", [round(x * 100, 1) for x in probs])
if q["template"] == "a":
    print("Q3 Number:", round(probs[0] * 100, 1))
else:
    kept = 0
    for x in probs:
        kept = kept + x
        if kept >= q["top_p"]:
            break
    print("Q3: kept mass", round(kept * 100, 1))
    print("Q3 Number:", round(probs[0] / kept * 100, 1))

# Q4
q = p3["q4"]
if q["template"] == "a":
    wrong_x = 100 - q["c_x"]
    wrong_y = 100 - q["c_y"] - q["a_y"]
    score_x = q["c_x"] - q["penalty"] * wrong_x
    score_y = q["c_y"] - q["penalty"] * wrong_y
    print("Q4: X =", score_x, " Y =", score_y)
    print("Q4 Number:", score_y - score_x)
else:
    questions = 0
    wrong = 0
    for s in q["sets"]:
        if s[3]:
            questions = questions + s[1]
            wrong = wrong + s[1] * s[2] / 100
    print("Q4 Number:", round(wrong / questions * 100, 1), "%")

# Q5
q = p3["q5"]
if q["template"] == "a":
    ratio = (q["tokens_b"] / q["params_b"]) / (1400 / 70)
    print("Q5 Number:", round(ratio, 2))
else:
    total_tokens = 0
    for i in range(1, q["turns"] + 1):
        total_tokens = total_tokens + q["per_turn"] * i
    print("Q5 Number:", total_tokens)

GitHub username: tomiris-nurbolat
Session code: PRACTICE
Fingerprint: bfd9a65e

PART 1:

1a
| A | 0.583 | 0.683 | 0.767 | 0.643 |
| B | 0.567 | 0.683 | 0.733 | 0.637 |

1b
| A, k=3 | 0.683 | 0.558 | 0.787 |
| A, k=5 | 0.767 | 0.646 | 0.856 |
| B, k=3 | 0.683 | 0.558 | 0.787 |
| B, k=5 | 0.733 | 0.61 | 0.829 |

1c
| A, k=3 | 19084 |
| A, k=5 | 24764 |
| B, k=3 | 13115 |
| B, k=5 | 14815 |

Budget: 21900

Recall by language (for 1d)
A, k=3:  EN 16/20 [0.58, 0.92]  RU 13/20 [0.43, 0.82]  KK 12/20 [0.39, 0.78]
A, k=5:  EN 17/20 [0.64, 0.95]  RU 15/20 [0.53, 0.89]  KK 14/20 [0.48, 0.85]
B, k=3:  EN 16/20 [0.58, 0.92]  RU 13/20 [0.43, 0.82]  KK 12/20 [0.39, 0.78]
B, k=5:  EN 16/20 [0.58, 0.92]  RU 14/20 [0.48, 0.85]  KK 14/20 [0.48, 0.85]

PART 2:
| H | 0.9 | KK, degraded | 0.833 | 0.664 | 0.927 | 12266 |
| L | 0.778 | RU, degraded | 0.7 | 0.521 | 0.833 | 10800 |

Personal-data pages per year: 180675

All cells (for 2b)
H EN clean 29/30 = 0.967 [0.833, 0.994]
H EN degraded 27/30 = 0.9 [0.744

`ranks` and `docs` are ready for you to compute with. Remember the **fingerprint** at the top of the data sheet: it goes into the first lines of `ANSWERS.md`.